# Backpropagation for Regression — From Scratch

**Goal:** Student ka `cgpa` ($x_1$) aur `profile_score` ($x_2$) dekh kar unka salary package `lpa` ($y$) predict karna.

**Network Structure:**
- **Input Layer:** 2 nodes ($x_1, x_2$)
- **Hidden Layer:** 2 nodes ($o_{11}, o_{12}$) — weights: `w111, w121, b11` aur `w112, w122, b12`
- **Output Layer:** 1 node ($y_{hat}$) — weights: `w211, w221, b21`

## Step 1 — Dataset

In [126]:
import numpy as np
import pandas as pd

df = pd.DataFrame(
    [[8, 8, 4],
     [7, 9, 5],
     [6, 10, 6],
     [5, 12, 7]],
    columns=['cgpa', 'profile_score', 'lpa']
)

df

,cgpa,profile_score,lpa
0,8,8,4
1,7,9,5
2,6,10,6
3,5,12,7


## Step 2 — Initialize Weights

Sab weights ko `0.1` aur biases ko `0.0` se initialize karte hain.

In [127]:
def initialize_weights():
    return {
        # Layer 1 (Input -> Hidden)
        'w111': 0.1, 'w121': 0.1, 'b11': 0.0,   # Hidden node 1 ke weights
        'w112': 0.1, 'w122': 0.1, 'b12': 0.0,   # Hidden node 2 ke weights

        # Layer 2 (Hidden -> Output)
        'w211': 0.1, 'w221': 0.1, 'b21': 0.0    # Output node ke weights
    }

weights = initialize_weights()
weights

{'w111': 0.1,
 'w121': 0.1,
 'b11': 0.0,
 'w112': 0.1,
 'w122': 0.1,
 'b12': 0.0,
 'w211': 0.1,
 'w221': 0.1,
 'b21': 0.0}

## Step 3 — Forward Propagation (Single Neuron Concept)

Ek **single neuron** ka output nikalne ka rule hume pata hai:
$$\text{output} = \sum (w_i \cdot x_i) + b = \text{np.dot}(\text{inputs}, \text{weights}) + \text{bias}$$

Pehle yeh single neuron function banate hain, phir poore network me use karte hain.

In [3]:
def linear_forward(inputs, node_weights, bias):
    """
    Kisi bhi single neuron ka linear output: z = (inputs . weights) + bias
    """
    return np.dot(inputs, node_weights) + bias


def forward(x1, x2, weights):
    inputs = [x1, x2]

    # 1. Hidden Node 1 ka output (o11)
    w_node1 = [weights['w111'], weights['w121']]
    o11 = linear_forward(inputs, w_node1, weights['b11'])

    # 2. Hidden Node 2 ka output (o12)
    w_node2 = [weights['w112'], weights['w122']]
    o12 = linear_forward(inputs, w_node2, weights['b12'])

    # 3. Output Node ka output (y_hat)
    w_output = [weights['w211'], weights['w221']]
    y_hat = linear_forward([o11, o12], w_output, weights['b21'])

    return o11, o12, y_hat


# Sample 0 test karo (x1=8, x2=8, actual y=4)
o11, o12, y_hat = forward(8, 8, weights)
print(f"o11 (Hidden 1) = {o11}")
print(f"o12 (Hidden 2) = {o12}")
print(f"y_hat (Output) = {y_hat}")

o11 (Hidden 1) = 1.6
o12 (Hidden 2) = 1.6
y_hat (Output) = 0.32000000000000006


## Step 4 — Backward Propagation (Gradients & Update)

1. **Step 1:** Gradients nikalo (chain rule se).
2. **Step 2:** Weights update karo: `weight = weight - (lr * gradient)`.

In [4]:
def backward(x1, x2, y, y_hat, o11, o12, weights, lr=0.001):
    # Error slope: dL/dy_hat (Loss = (y - y_hat)^2)
    dL_dyhat = -2 * (y - y_hat)

    # -----------------------------------------------------
    # STEP 1: Gradients (Derivatives) nikalo
    # -----------------------------------------------------
    # Output layer
    d_w211 = dL_dyhat * o11
    d_w221 = dL_dyhat * o12
    d_b21  = dL_dyhat * 1

    # Hidden layer (Chain rule: error * output_weight * input)
    d_w111 = dL_dyhat * weights['w211'] * x1
    d_w121 = dL_dyhat * weights['w211'] * x2
    d_b11  = dL_dyhat * weights['w211'] * 1

    d_w112 = dL_dyhat * weights['w221'] * x1
    d_w122 = dL_dyhat * weights['w221'] * x2
    d_b12  = dL_dyhat * weights['w221'] * 1

    # -----------------------------------------------------
    # STEP 2: Weights Update karo (w = w - lr * gradient)
    # -----------------------------------------------------
    weights['w211'] -= lr * d_w211
    weights['w221'] -= lr * d_w221
    weights['b21']  -= lr * d_b21

    weights['w111'] -= lr * d_w111
    weights['w121'] -= lr * d_w121
    weights['b11']  -= lr * d_b11

    weights['w112'] -= lr * d_w112
    weights['w122'] -= lr * d_w122
    weights['b12']  -= lr * d_b12

# Ek sample update test
print("w211 before update:", weights['w211'])
backward(8, 8, y=4, y_hat=y_hat, o11=o11, o12=o12, weights=weights, lr=0.001)
print("w211 after update :", weights['w211'])

w211 before update: 0.1
w211 after update : 0.111776


## Step 5 — Training Loop

Har epoch me:
1. `forward()` $\rightarrow$ prediction nikalo
2. `loss` calculate karo
3. `backward()` $\rightarrow$ weights update karo

In [5]:
def train(df, epochs=200, lr=0.001):
    weights = initialize_weights()

    print(f"Training start: Epochs = {epochs}, Learning Rate = {lr}\n")
    print(f"{'Epoch':>6} | {'Average Loss (MSE)':>20}")
    print("-" * 30)

    for epoch in range(1, epochs + 1):
        losses = []

        for i in range(len(df)):
            x1 = df['cgpa'].values[i]
            x2 = df['profile_score'].values[i]
            y  = df['lpa'].values[i]

            # 1. Forward Pass
            o11, o12, y_hat = forward(x1, x2, weights)

            # 2. Loss calculate karo
            loss = (y - y_hat) ** 2
            losses.append(loss)

            # 3. Backward Pass (Weights Update)
            backward(x1, x2, y, y_hat, o11, o12, weights, lr)

        if epoch == 1 or epoch == 5 or epoch % 20 == 0 or epoch == epochs:
            print(f"{epoch:>6} | {np.mean(losses):>20.6f}")

    print("\nTraining complete!")
    return weights

## Step 6 — Run Training

In [6]:
# Yahan aap epochs ya lr change kar sakte hain:
EPOCHS = 200
LR = 0.001

trained_weights = train(df, epochs=EPOCHS, lr=LR)

Training start: Epochs = 200, Learning Rate = 0.001

 Epoch |   Average Loss (MSE)
------------------------------
     1 |            26.345578
     5 |             1.223804
    20 |             0.698711
    40 |             0.429469
    60 |             0.258872
    80 |             0.156454
   100 |             0.099548
   120 |             0.070250
   140 |             0.056023
   160 |             0.049335
   180 |             0.046206
   200 |             0.044713

Training complete!


## Step 7 — Final Predictions vs Actual

In [7]:
print(f"{'Student':>8} | {'CGPA':>6} | {'Profile':>8} | {'Actual LPA':>11} | {'Predicted LPA':>14}")
print("-" * 58)

for i in range(len(df)):
    x1 = df['cgpa'].values[i]
    x2 = df['profile_score'].values[i]
    y  = df['lpa'].values[i]

    _, _, y_hat = forward(x1, x2, trained_weights)
    print(f"{i:>8} | {x1:>6} | {x2:>8} | {y:>11} | {y_hat:>14.4f}")

 Student |   CGPA |  Profile |  Actual LPA |  Predicted LPA
----------------------------------------------------------
       0 |      8 |        8 |           4 |         4.2177
       1 |      7 |        9 |           5 |         4.9694
       2 |      6 |       10 |           6 |         5.7210
       3 |      5 |       12 |           7 |         7.1053
